In [ ]:
import gzip
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="YlOrRd", style="whitegrid", font_scale=1)
sns.color_palette("YlOrRd", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Galmuri11'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
vcf_path = 'data/clinvar_20260208.vcf'

# 일단 다 봅시다

In [ ]:
cnt = 0
with open(vcf_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'): continue
        cols = line.strip().split('\t')
        print(cols)
        cnt += 1
        if cnt >= 5: break

# 뭐야 이거 어떻게 읽어요?
>['1', '66926', '3385321', 'AG', 'A', '.', '.', 'ALLELEID=3544463;CLNDISDB=Human_Phenotype_Ontology:HP:0000547,MONDO:MONDO:0019200,MeSH:D012174,MedGen:C0035334,OMIM:268000,OMIM:PS268000,Orphanet:791;CLNDN=Retinitis_pigmentosa;CLNHGVS=NC_000001.10:g.66927del;CLNREVSTAT=criteria_provided,_single_submitter;CLNSIG=Uncertain_significance;CLNSIGSCV=SCV005419006;CLNVC=Deletion;CLNVCSO=SO:0000159;GENEINFO=OR4F5:79501;MC=SO:0001627|intron_variant;ORIGIN=0']

## 위치 정보
1. 1, 66926: 1번 염색체 66926번째 위치
2. 3385321: Clinvar 아이디
3. 'AG' / 'A': AG가 A로 바뀐 돌연변이 ~~염기 하나 가출함~~
4. 점 두개: 각각 QUAL, FILTER

## 핵심 분석 정보
- ALLELEID 이후 구역입니다.
1. ALLELEID: 대립 유전자 아이디
2. CLNDISDB: 질병 데이터베이스 링크
3. CLNDN: 질병 이름
4. CLNHGVS: HGVS 명명법
5. CLNREVSTAT: 검토 신뢰도
6. CLNSIG: 임상적 유의성
7. CLNVC: 변이 타입
8. GENEINFO: 유전자 정보(이름:Entrez ID)
9. MC: 분자적 영향
10. ORIGIN: 변이의 기원(0: 특정되지 않음)

# 임상적 유의성이 Pathogenic인 것만 (5개)

In [ ]:
vcf_path = 'data/clinvar_20260208.vcf'

cnt = 0
with open(vcf_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'): continue

        cols = line.strip().split('\t')
        chrom, pos, id_var, ref, alt = cols[0], cols[1], cols[2], cols[3], cols[4]
        info = cols[7]

        if 'CLNSIG=Pathogenic' in info:
            # 서열이 20자보다 길면 '...'으로 줄여서 출력하기
            display_ref = ref if len(ref) <= 20 else f"{ref[:17]}..."
            display_alt = alt if len(alt) <= 20 else f"{alt[:17]}..."

            print(f"ID: {id_var} | Pos: {chrom}:{pos} | Ref/Alt: {display_ref}/{display_alt}")

            # INFO 필드에서 핵심 정보(유전자 이름 등)만 뽑아서 보기
            gene_info = [x for x in info.split(';') if x.startswith('GENEINFO=')]
            gene_name = gene_info[0].split(':')[0].replace('GENEINFO=', '') if gene_info else "Unknown"
            print(f"  👉 Gene: {gene_name}")
            print("-" * 50)

            cnt += 1

        if cnt >= 10: break

# Significance countplot

In [ ]:
rows = []

with open(vcf_path, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        cols = line.strip().split('\t')

        # INFO 파싱
        info_dict = dict(item.split('=') for item in cols[7].split(';') if '=' in item)

        # 분석에 필요한 열만 추출
        rows.append({
            'Chrom': cols[0],
            'Pos': int(cols[1]),
            'Type': info_dict.get('CLNVC', 'Unknown'),
            'Significance': info_dict.get('CLNSIG', 'Unknown').split(',')[0], # 여러 개일 경우 첫 번째 것만
            'Gene': info_dict.get('GENEINFO', 'Unknown').split(':')[0]
        })

df = pd.DataFrame(rows)

def clean_sig(sig):
    sig = sig.lower()
    if 'pathogenic' in sig and 'likely' not in sig: return 'Pathogenic'
    if 'likely pathogenic' in sig: return 'Likely Pathogenic'
    if 'benign' in sig and 'likely' not in sig: return 'Benign'
    if 'likely benign' in sig: return 'Likely Benign'
    if 'uncertain' in sig: return 'VUS (Uncertain)'
    return 'Others' # 너무 복잡한 것들은 다 여기로

df['Sig_Clean'] = df['Significance'].apply(clean_sig)

# 시각화: 임상적 유의성 분포
plt.figure(figsize=(25, 8))
sns.countplot(x='Sig_Clean', data=df, hue = 'Sig_Clean', palette="YlOrRd")
# df['Significance'].value_counts().plot(kind='bar', color='skyblue')
plt.title('ClinVar 변이의 임상적 유의성 분포')
plt.xlabel('유의성 (CLNSIG)')
plt.ylabel('변이 개수')
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 염색체별 Pathogenic 분포

In [ ]:
# 1. Pathogenic(병원성) 데이터만 필터링
# (앞서 만든 Sig_Clean 컬럼이 있다고 가정하거나, 원본에서 필터링)
pathogenic_df = df[df['Significance'].str.contains('Pathogenic', case=False, na=False)]

# 2. 염색체 순서 정렬을 위한 가이드 (1~22, X, Y 순서)
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']
pathogenic_df['Chrom'] = pd.Categorical(pathogenic_df['Chrom'], categories=chrom_order, ordered=True)

# 3. 염색체별 개수 집계
chrom_counts = pathogenic_df['Chrom'].value_counts().sort_index()

# 4. 시각화
plt.figure(figsize=(15, 7))
colors = plt.cm.get_cmap('YlOrRd')(np.linspace(0.4, 0.9, len(chrom_counts))) # 빨간색 계열로 강조

chrom_counts.plot(kind='bar', color=colors, edgecolor='black', linewidth=1)

plt.title('염색체별 Pathogenic(병원성) 변이 분포', fontsize=18, pad=20)
plt.xlabel('염색체 번호', fontsize=13)
plt.ylabel('병원성 변이 개수', fontsize=13)
plt.xticks(rotation=0) # 숫자는 똑바로 서있어야 제맛
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 막대 위에 숫자 표시 (가독성 업그레이드)
for i, v in enumerate(chrom_counts):
    plt.text(i, v + 500, f'{int(v):,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

- 2, 17, 1번이 TOP 3인듯... Y염색체가 제일 적다.
- 뭔가 잘못된게 아니고 원래 Y염색체가 좀 작아요. 들어있는 유전자도 적음...

## TOP 3의 유전자들

### 2번 염색체

In [ ]:
# 1. 2번 염색체의 Pathogenic 데이터만 추출
chr2_pathogenic = df[(df['Chrom'] == '2') & (df['Significance'].str.contains('Pathogenic', case=False, na=False))]

# 2. 유전자별 변이 개수 집계
top10_genes_chr2 = chr2_pathogenic['Gene'].value_counts().head(10)

# 3. 시각화 (NanumSquare 적용)
plt.figure(figsize=(12, 8))
colors = plt.cm.get_cmap('Oranges_r')(np.linspace(0.2, 0.7, 10))

top10_genes_chr2.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.title('2번 염색체 내 Pathogenic 변이 상위 10개 유전자', fontsize=16, pad=20)
plt.xlabel('병원성 변이 개수', fontsize=12)
plt.ylabel('유전자 이름 (Gene Symbol)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 막대 끝에 숫자 표시
for i, v in enumerate(top10_genes_chr2.sort_values()):
    plt.text(v + 10, i, f'{int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 17번 염색체

In [ ]:
# 1. 2번 염색체의 Pathogenic 데이터만 추출
chr17_pathogenic = df[(df['Chrom'] == '17') & (df['Significance'].str.contains('Pathogenic', case=False, na=False))]

# 2. 유전자별 변이 개수 집계
top10_genes_chr17 = chr17_pathogenic['Gene'].value_counts().head(10)

# 3. 시각화 (NanumSquare 적용)
plt.figure(figsize=(12, 8))
colors = plt.cm.get_cmap('Oranges_r')(np.linspace(0.2, 0.7, 10))

top10_genes_chr17.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.title('17번 염색체 내 Pathogenic 변이 상위 10개 유전자', fontsize=16, pad=20)
plt.xlabel('병원성 변이 개수', fontsize=12)
plt.ylabel('유전자 이름 (Gene Symbol)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 막대 끝에 숫자 표시
for i, v in enumerate(top10_genes_chr17.sort_values()):
    plt.text(v + 10, i, f'{int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 1번 염색체

In [ ]:
# 1. 2번 염색체의 Pathogenic 데이터만 추출
chr1_pathogenic = df[(df['Chrom'] == '1') & (df['Significance'].str.contains('Pathogenic', case=False, na=False))]

# 2. 유전자별 변이 개수 집계
top10_genes_chr1 = chr1_pathogenic['Gene'].value_counts().head(10)

# 3. 시각화 (NanumSquare 적용)
plt.figure(figsize=(12, 8))
colors = plt.cm.get_cmap('Oranges_r')(np.linspace(0.2, 0.7, 10))

top10_genes_chr1.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.title('1번 염색체 내 Pathogenic 변이 상위 10개 유전자', fontsize=16, pad=20)
plt.xlabel('병원성 변이 개수', fontsize=12)
plt.ylabel('유전자 이름 (Gene Symbol)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 막대 끝에 숫자 표시
for i, v in enumerate(top10_genes_chr1.sort_values()):
    plt.text(v + 10, i, f'{int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 성염색체
- 위에 있는거 다 해보면 안되냐고요? 22개 다 감당하실 수 있겠어요? 

### 마이너의 메이저-Y염색체

In [ ]:
# 1. 2번 염색체의 Pathogenic 데이터만 추출
chry_pathogenic = df[(df['Chrom'] == 'Y') & (df['Significance'].str.contains('Pathogenic', case=False, na=False))]

# 2. 유전자별 변이 개수 집계
top10_genes_chry = chry_pathogenic['Gene'].value_counts().head(10)

# 3. 시각화 (NanumSquare 적용)
plt.figure(figsize=(12, 8))
colors = plt.cm.get_cmap('Oranges_r')(np.linspace(0.2, 0.7, 10))

top10_genes_chry.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.title('Y 염색체 내 Pathogenic 변이 상위 10개 유전자', fontsize=16, pad=20)
plt.xlabel('병원성 변이 개수', fontsize=12)
plt.ylabel('유전자 이름 (Gene Symbol)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 막대 끝에 숫자 표시
for i, v in enumerate(top10_genes_chry.sort_values()):
    plt.text(v + 1, i, f'{int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### X염색체

In [ ]:
# 1. 2번 염색체의 Pathogenic 데이터만 추출
chrx_pathogenic = df[(df['Chrom'] == 'X') & (df['Significance'].str.contains('Pathogenic', case=False, na=False))]

# 2. 유전자별 변이 개수 집계
top10_genes_chrx = chrx_pathogenic['Gene'].value_counts().head(10)

# 3. 시각화 (NanumSquare 적용)
plt.figure(figsize=(12, 8))
colors = plt.cm.get_cmap('Oranges_r')(np.linspace(0.2, 0.7, 10))

top10_genes_chrx.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.title('X 염색체 내 Pathogenic 변이 상위 10개 유전자', fontsize=16, pad=20)
plt.xlabel('병원성 변이 개수', fontsize=12)
plt.ylabel('유전자 이름 (Gene Symbol)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 막대 끝에 숫자 표시
for i, v in enumerate(top10_genes_chrx.sort_values()):
    plt.text(v + 10, i, f'{int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 염색체 그런건 모르겠고 제가 왕입니다
- 니가 왕이면 안되지...

In [ ]:
# 1. 전체 데이터에서 Pathogenic 변이만 필터링
all_pathogenic = df[df['Significance'].str.contains('Pathogenic', case=False, na=False)]

# 2. 유전자별 변이 개수 집계 (전체 통합 Top 20)
top20_genes = all_pathogenic['Gene'].value_counts().head(20)

# 3. 시각화 (NanumSquare 폰트 설정 반영)
plt.figure(figsize=(14, 10))
# 데이터의 강도를 표현하기 위해 점진적인 색상 변화 적용
colors = plt.cm.get_cmap('magma')(np.linspace(0.3, 0.8, 20))

top20_genes.sort_values().plot(kind='barh', color=colors, edgecolor='black', alpha=0.9)

plt.title('ClinVar 전체 통합 Pathogenic 변이 TOP 20 유전자', fontsize=20, pad=25)
plt.xlabel('병원성 변이(Pathogenic) 누적 개수', fontsize=14)
plt.ylabel('유전자 심볼 (Gene Symbol)', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.4)

# 막대 끝에 개수 표시 (천 단위 쉼표 포함)
for i, v in enumerate(top20_genes.sort_values()):
    plt.text(v + 100, i, f'{int(v):,}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# 어떤 변이가 제일 많은가

In [ ]:
# 1. 데이터 집계
variant_counts = df['Type'].value_counts().reset_index()
variant_counts.columns = ['Variant Type', 'Count']

# 2. 시각화
plt.figure(figsize=(14, 8))

# Seaborn의 컬러 팔레트 활용 (viridis, rocket, mako 등)
ax = sns.barplot(data=variant_counts, x='Count', y='Variant Type',
                 palette='viridis', edgecolor='black')

plt.title('ClinVar 변이 타입별 전체 분포 (Seaborn Ver.)', fontsize=18, pad=20)
plt.xlabel('변이 개수', fontsize=12)
plt.ylabel('변이 종류', fontsize=12)

# 각 막대 끝에 숫자 표시
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}',
                (p.get_width(), p.get_y() + p.get_height()/2),
                ha='left', va='center', xytext=(5, 0),
                textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.show()

- single nucleotide variant: 염기 하나가 바뀜
- 찾기가 진짜 빡셉니다.

## SNV가 가장 많은 염색체는?

In [ ]:
# 1. SNV 데이터만 필터링
snv_df = df[df['Type'] == 'single_nucleotide_variant']

# 2. 염색체 순서 정렬 (1~22, X, Y)
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

# 3. 시각화
plt.figure(figsize=(16, 8))

# countplot으로 염색체별 빈도 계산
ax = sns.countplot(data=snv_df, x='Chrom', order=chrom_order, palette='magma')

plt.title('염색체별 SNV(Single Nucleotide Variant) 발생 빈도', fontsize=18, pad=20)
plt.xlabel('염색체 번호', fontsize=12)
plt.ylabel('SNV 개수', fontsize=12)

# 막대 위에 숫자 표시
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 7),
                textcoords='offset points')

plt.tight_layout()
plt.show()

- 1번이 제일 많군...

## 그 중에 위험한 변이가 여기 있다

In [ ]:
# 1. Pathogenic 하면서 동시에 SNV인 데이터만 필터링
patho_snv_df = df[(df['Type'] == 'single_nucleotide_variant') &
                  (df['Significance'].str.contains('Pathogenic', case=False, na=False))]

# 2. 염색체 순서 정렬
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

# 3. 시각화
plt.figure(figsize=(16, 8))

# 막대 그래프 그리기
ax = sns.countplot(data=patho_snv_df, x='Chrom', order=chrom_order, palette='Reds_r')

plt.title('염색체별 Pathogenic SNV 분포 (진짜 빌런 찾기)', fontsize=18, pad=20)
plt.xlabel('염색체 번호', fontsize=12)
plt.ylabel('병원성 SNV 개수', fontsize=12)

# 각 막대 위에 숫자 표시
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=10, color='black', xytext=(0, 7),
                textcoords='offset points')

plt.tight_layout()
plt.show()

## pathogenic snv 비중이 높은 유전자들

In [ ]:
# 1. 유전자별 전체 SNV 개수 계산
gene_snv_total = df[df['Type'] == 'single_nucleotide_variant']['Gene'].value_counts()

# 2. 유전자별 Pathogenic SNV 개수 계산
gene_patho_snv = df[(df['Type'] == 'single_nucleotide_variant') &
                    (df['Significance'].str.contains('Pathogenic', case=False, na=False))]['Gene'].value_counts()

# 3. 데이터 합치기 (최소 보고 건수가 100건 이상인 유전자만 대상 - 너무 적으면 신뢰도 하락)
density_df = pd.DataFrame({'Total_SNV': gene_snv_total, 'Patho_SNV': gene_patho_snv}).fillna(0)
density_df = density_df[density_df['Total_SNV'] >= 100] # 최소 100건 이상 보고된 유전자만

# 4. 병원성 변이 밀도(비율) 계산
density_df['Patho_Ratio'] = (density_df['Patho_SNV'] / density_df['Total_SNV']) * 100

# 5. 비율 상위 10개 추출
top10_nasty_genes = density_df.sort_values(by='Patho_Ratio', ascending=False).head(15)

# 6. 시각화
plt.figure(figsize=(14, 8))

ax = sns.barplot(x=top10_nasty_genes['Patho_Ratio'], y=top10_nasty_genes.index, palette='flare')

plt.title('전체 SNV 중 Pathogenic 비중이 가장 높은 "독종" 유전자 TOP 10', fontsize=18, pad=20)
plt.xlabel('병원성 변이 비중 (%)', fontsize=12)
plt.ylabel('유전자 이름', fontsize=12)

# 막대 끝에 퍼센트 표시
for i, v in enumerate(top10_nasty_genes['Patho_Ratio']):
    plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()